# Youtube Data API: Search
https://developers.google.com/youtube/v3/docs/search

## Contenido:

* [Librerias](*Librerias)
* [Search](*Search)


<a name="Librerias"></a>
## Librerias

In [ ]:
# ========= Data Cleaning =========

!pip install advertools
!pip install clean-text
!pip install urlextract
import advertools as adv
from urlextract import URLExtract
from nltk.tokenize import regexp_tokenize
import html
import string
import re
from cleantext import clean
import unicodedata

# ========= Data Cleaning =========

import pandas as pd
import requests
import csv
import time
from datetime import datetime
from datetime import date
import os.path
from os import path
from google.colab import drive
key='TU_CLAVE_DE_API'
drive.mount('/content/drive/')

#### https://www.youtube.com/playlist?list=
#### https://www.youtube.com/channel/
#### https://www.youtube.com/watch?v=

## Search

### Ejecucion

#### Parametros constantes

In [ ]:
URIs='https://www.googleapis.com/youtube/v3/search?key='

# ================== Parametros Requeridos ==================

part = 'snippet' # Parametro obligatorio

# ================== Filtros (especifique 0 o 1 de los siguientes parámetros) ==================

# relatedToVideoId = ID de un video para obtener resultados relacionados con este video (type=video)

# ================== Parametros opcionales ==================

# channelId = Indicar que la respuesta solo debe tener resultados de este canal
channelType = 'any' # Restringir la busqueda a un tipo de canal: any devuelve todos los canales
# eventType = Restringe la busqueda a eventos de transmision (type=video)
# location = Define las coordenadas latitud/longitud para buscar en esta ubicacion (type= video)
# locationRadius = Radio con centro en location
maxResults = '10' # Numero maximo de elementos a devolver. El rango es 0 a 50.
order = 'viewCount' # Metodo para ordenar los resultados. date = orden cronologico inverso, ====> OJO Revisar
# rating = se ordenan de mayor a menor calificacion, relevance = se ordenan segun su relevancia (predeterminado),
# title = alfabeticamente por titulo, videCount = orden descendente de su numero de videos subidos,
# viewCount = de mayor a menor numero de vistas.

# publishedAfter = Solo debe tener resultados en el momento especificado o despues
# publishedBefore = Solo debe tener resultados en el momento especificado o antes

relevanceLanguage = 'es' # Devuelve resultados de busqueda mas relevantes para el idioma especificado (Codigo 2 letras)

safeSearch = 'none' # Indica si debe contener contenido restringido aparte del contenido estandar.
# moderate = no lo restringe del todo (predeterminado), none = no filtra ninguno, strict = filtra todo el
# contenido restringido
# type = recupera un tipo partcilar de recurso video, channel o playlist el valor predeterminado es video,channel,playlist
# videoCategoryId = filtra resultados de busqueda de videos segun su categoria (type=video)
# videoType = restringe busqueda a un tipo particuclar de videos (type=video), los valores son any = todos los videos,
# episode = episodios de programas, movie = solo peliculas

#### Parametros variables

In [ ]:
# ================== Parametros opcionales ==================

pageToken = '' # Pagina especifica que se debe devolver
q = 'Proyecto minero quebradona ' # Query
regionCode = 'CO' # Devuelve resultados que se pueden ver en el pais especificado

#### Solicitud

In [ ]:
r = requests.get(URIs+key+'&part='+part+'&channelType='+channelType\
                 +'&maxResults='+maxResults+'&order='+order\
                 +'&pageToken='+pageToken+'&q='+q\
                 +'&regionCode='+regionCode+'&relevanceLanguage='+relevanceLanguage\
                 +'&safeSearch='+safeSearch)

In [ ]:
r

In [ ]:
r.reason

In [ ]:
d = dict(r.json())

In [ ]:
d.keys()

In [ ]:
d

In [ ]:
d['kind']

In [ ]:
d['etag']

In [ ]:
d['nextPageToken']

In [ ]:
d['regionCode']

In [ ]:
d['pageInfo']

#### Resultados

In [ ]:
items = d.get('items')
items

#### Dataframe

In [ ]:
data=pd.json_normalize(items)
data.info()

In [ ]:
data

### Algoritmo en Produccion

#### Parametros constantes

In [ ]:
URIs = 'https://www.googleapis.com/youtube/v3/search?key='

# ================== Parametros Requeridos ==================

part = 'snippet' # Parametro obligatorio
fields = 'pageInfo(*),nextPageToken(*),prevPageToken(*),regionCode(*),\
items(id,snippet(title,description,publishTime,publishedAt,liveBroadcastContent,channelId,channelTitle))'

# ================== Parametros opcionales ==================

channelType = 'any' # Restringir la busqueda a un tipo de canal: any devuelve todos los canales
maxResults = '25' # Numero maximo de elementos a devolver. El rango es 0 a 50.
order = 'relevance' # Metodo para ordenar los resultados. date = orden cronologico inverso, ====> OJO Revisar
# rating = se ordenan de mayor a menor calificacion, relevance = se ordenan segun su relevancia (predeterminado),
# title = alfabeticamente por titulo, videCount = orden descendente de su numero de videos subidos,
# viewCount = de mayor a menor numero de vistas.
relevanceLanguage = 'es' # Devuelve resultados de busqueda mas relevantes para el idioma especificado (Codigo 2 letras)
safeSearch = 'none' # Indica si debe contener contenido restringido aparte del contenido estandar.
# moderate = no lo restringe del todo (predeterminado), none = no filtra ninguno, strict= filtra todo el
# contenido restringido

#### Operadores de busqueda

Identificar palabras clave

+palabraclave

-palabranorelevante

""

1..3 encontrar resultados dentro de este rango

and

or

* comidin para cualquier palabra

() combinar operadores.

#### Parametros variables

In [ ]:
'''
consultas=['serie "Pablo Escobar el patron del mal" trailer -narcos -cielos -sapos -muñecas -chema'\
           ,'Serie Escobar el patron del mal trailer -narcos -cielos -sapos -muñecas -chema -chapo'
           # capitulos
           ,'Serie Escobar el patron del mal capitulos -narcos -cielos -sapos -muñecas -chema -chapo'
           ,'serie "Pablo Escobar el patron del mal" capitulos -narcos -cielos -sapos -muñecas -chema'\
           # episodios
           ,'serie "Pablo Escobar el patron del mal" episodios -narcos -mexico -cielos -sapos -muñecas -chema'\
           ,'Serie Escobar el patron del mal episodios -narcos -mexico -cielos -sapos -muñecas -chema -chapo'
           # mejores escenas
           ,'serie "Pablo Escobar el patron del mal" mejores escenas -narcos -cielos -sapos -muñecas -chema'
           # escenas
           ,'serie colombiana "Pablo Escobar el patron del mal" escenas -narcos -mexico -cielos -sapos -muñecas -chema'\
           ,'Serie Escobar el patron del mal escenas -narcos -mexico -cielos -sapos -muñecas -chema -chapo'
           # ver mejores escenas
           ,'ver serie "Pablo Escobar el patron del mal" mejores escenas -narcos -cielos -sapos -muñecas -chema'\
           # mejores escenas
           ,'serie "Escobar el patron del mal" mejores escenas -narcos -cielos -sapos -muñecas -chema'\
           # otros
           ,'Serie Escobar el patron del mal -narcos -cielos -sapos -muñecas -chema -chapo'
           ,'Serie Pablo Escobar, el Patron del mal -narcos -cielos -sapos -muñecas -chema -chapo'
           ,'"serie Pablo Escobar el patron del mal" -narcos -cielos -sapos -muñecas -chema'\
           ,'serie "Pablo Escobar, el patron del mal" -narcos -cielos -sapos -muñecas -chema'\
           ,'serie "Escobar, el patron del mal" -narcos -cielos -sapos -muñecas -chema'\
           ,'serie "Escobar el patron del mal" colombia -narcos -mexico -cielos -sapos -muñecas -chema'\
           ,'serie "Escobar el patron del mal" Caracol television -narcos -cielos -sapos -muñecas -chema'] # Querys OJO MODIFICAR LISTA PARA SERIE Y/O TELENOVELA ==Escobar, el patron del mal==

consultas=['serie "Narcos Netflix" temporadas 1..3 -mexico'\
           # temporada
           ,'serie "Narcos Netflix" temporada 1 -mexico'\
           # temporadas
           ,'serie "Narcos Netflix" temporadas -mexico'\
           # trailer
           ,'serie "Narcos Netflix" trailer -mexico'\
           # capitulos
           ,'serie "Narcos Netflix" colombia capitulos -mexico'\
           # episodios
           ,'serie "Narcos Netflix" episodios -mexico'\
           # mejores escenas
           ,'serie "Narcos Netflix" mejores escenas -mexico'\
           # escenas
           ,'serie "Narcos Netflix" colombia escenas -mexico'\
           # ver mejores escenas
           ,'ver serie "Narcos Netflix" mejores escenas -mexico'\
           # otros
           ,'serie "Narcos" Netflix -mexico'\
           ,'serie "Narcos de Netflix" -mexico'\
           ,'"serie Narcos Netflix" -mexico'\
           ,'serie "Narcos Netflix" colombia -mexico'\
           ,'serie "Narcos Netflix" narcotrafico colombia -mexico'] # Querys OJO MODIFICAR LISTA PARA SERIE Y/O TELENOVELA ==NARCOS==

consultas=['serie "Tres Caines" trailer'\
           # trailer
           ,'serie "3 caines" trailer'\
           # capitulos
           ,'serie "Tres Caines" colombia capitulos'\
           ,'serie "3 caines" colombia capitulos'\
           ,'serie de television colombiana "Tres Caines" capitulos'\
           ,'serie de television colombiana "3 Caines" capitulos'\
           # episodios
           ,'serie "Tres Caines" canal rcn episodios'\
           ,'serie "3 Caines" canal rcn episodios'\
           ,'serie de television colombiana "Tres Caines" episodios'\
           ,'serie de television colombiana "3 Caines" episodios'\
           # mejores escenas
           ,'serie "Tres Caines" mejores escenas'\
           ,'serie "3 Caines" mejores escenas'\
           ,'serie "3 caines" canal rcn mejores escenas'\
           # escenas
           ,'serie "Tres Caines" escenas'\
           ,'serie "3 Caines" escenas'\
           # ver mejores escenas
           ,'ver serie "Tres Caines" mejores escenas'\
           ,'ver telenovela "3 Caines" mejores escenas'\
           #otros
           ,'ver telenovela tres caines canal rcn'\
           ,'ver telenovela tres caines'\
           ,'serie tres caines'\
           ,'serie "3 caines canal rcn"'\
           ,'serie "Tres Caines" hermanos castaño canal rcn'\
           ,'serie "3 Caines" hermanos castaño canal rcn'\
           ,'"tres caines" canal rcn colombia'\
           ,'novela "tres caines" canal rcn'\
           ,'novela "3 caines" canal rcn'\
           ,'serie de television colombiana "Tres Caines" canal rcn'\
           ,'serie "Tres Caines"'\
           ,'serie "los tres caines"'\
           ,'serie "los 3 caines"'\
           ,'"los tres caines"'] # Querys OJO MODIFICAR LISTA PARA SERIE Y/O TELENOVELA ==Tres Caines==

consultas=['"serie chilena los 80" trailer'\
           ,'serie chilena "los 80" trailer'\
           # temporadas
           ,'serie chilena "los 80" temporadas'\
           ,'serie chilena los 80 temporadas '\
           ,'serie chilena "los 80" temporadas 1..7 -rockers -masterchef'\
           ,'serie chilena "los 80\'s" temporadas'\
           # capitulos
           ,'serie Chile "los 80" capitulos'\
           # episodios
           ,'serie chilena "los 80" episodios -animados'\
           # mejores escenas
           ,'serie chilena "los 80" mejores escenas'\
           # escenas
           ,'serie chilena "los 80" escenas -prisioneros'\
           # ver mejores escenas
           ,'ver serie chilena "los 80" mejores escenas'\
           # otros
           ,'serie chilena "los 80" canal 13 -caricaturas'\
           ,'serie chilena "los 80" mejores momentos'
           ,'"los 80 mas que una moda"'\
           ,'serie los 80 mas que una moda'\
           ,'serie de television chilena "los 80" -animados -caricaturas -años -decada'\
           ,'serie chilena los 80'\
           ,'serie chilena "los 80" mas que una moda'\
           ,'serie chilena los 80 mas que una moda'] # Querys OJO MODIFICAR LISTA PARA SERIE Y/O TELENOVELA ==los 80==

consultas = ['serie Dignidad amazon prime video trailer'\
             ,'serie Dignity amazon prime video trailer'\
             ,'serie "Dignidad" amazon prime video trailer'\
             ,'serie "dignidad" trailer amazon prime video'\
             # capitulos
             ,'serie dignidad prime video capitulos'\
             ,'serie Dignidad capitulos amazon prime'\
             # episodios
             ,'serie dignidad prime video episodios'\
             ,'serie Dignidad episodios amazon prime video'\
             # mejores escenas
             ,'serie dignidad prime video mejores escenas'\
             #escenas
             ,'serie dignidad prime video escenas'\
             # otros
             ,'serie dignidad amazon prime'\
             ,'serie dignidad amazon'
             ,'serie chilena Dignidad Prime video'] # Querys OJO MODIFICAR LISTA PARA SERIE Y/O TELENOVELA ==Dignidad==
'''

'''
paises = ['CL','DE','US'] # Devuelve resultados que se pueden ver en el pais especificado (Serie Dignity)
paises = ['CL','US'] # Devuelve resultados que se pueden ver en el pais especificado (Serie los 80)
paises = ['CO','US'] # Devuelve resultados que se pueden ver en el pais especificado (Serie Pablo Escobar, Tres Caines, Narcos)
'''
# ================== Parametros opcionales ==================

# ================== OJO modificar para cada serie ==================

consultas=['serie "Narcos Netflix" temporadas 1..3 -mexico'\
           # temporada
           ,'serie "Narcos Netflix" temporada 1 -mexico'\
           # temporadas
           ,'serie "Narcos Netflix" temporadas -mexico'\
           # trailer
           ,'serie "Narcos Netflix" trailer -mexico'\
           # capitulos
           ,'serie "Narcos Netflix" colombia capitulos -mexico'\
           # episodios
           ,'serie "Narcos Netflix" episodios -mexico'\
           # mejores escenas
           ,'serie "Narcos Netflix" mejores escenas -mexico'\
           ,'serie Narcos en netflix mejores escenas -mexico -cielos'\
           # escenas
           ,'serie "Narcos Netflix" colombia escenas -mexico'\
           # ver mejores escenas
           ,'ver serie "Narcos Netflix" mejores escenas -mexico'\
           # otros
           ,'serie Narcos netflix -mexico -cielos'\
           ,'serie "Narcos" Netflix -mexico'\
           ,'serie "Narcos de Netflix" -mexico'\
           ,'"serie Narcos Netflix" -mexico'\
           ,'serie "Narcos Netflix" colombia -mexico'\
           ,'serie "Narcos Netflix" narcotrafico colombia -mexico'] # Querys OJO MODIFICAR LISTA PARA SERIE Y/O TELENOVELA ==NARCOS==

paises = ['CO','US'] # Devuelve resultados que se pueden ver en el pais especificado (Serie Pablo Escobar, Tres Caines, Narcos)

# ================== OJO modificar para cada serie ==================

In [ ]:
len(consultas)

#### Carpetas Drive

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Search/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'SEARCH-Relevance'+str(datetime.now()) # +'Relevance' ========> OJO modificar si es por relevancia o por numero de vistas

#### Constantes

In [ ]:
solicitud = 0 # Inicio de las solicitudes

# ======================= OJO Cambiar cuando el parametro es por relevancia paginasmax=2 =======================

paginasmax = 1 # Numero de paginas maximas

# ======================= OJO Cambiar cuando el parametro es por relevancia paginasmax=2 =======================

solicitudmax = 100 # Numero de solicitudes maximas (el limite es 100, pues cada solicitud cuesta 100 unidades y el maximo son 10000 unidades diarias.)
espera = 0.5 # Valor de retardo para las solicitudes

# ======================= OJO Cambiar diariamente =======================

ithConsulta = 14 # i-esima Consulta
ithPais = 0 # i-esimo Pais

# ======================= OJO Cambiar diariamente =======================

#### Ejecucion

In [ ]:
Datac= pd.DataFrame() # Dataframe de Resultados Totales
Metadatac = pd.DataFrame(columns=['Solicitud', 'maxResultados', 'Consulta', 'Pais', 'Lenguaje', 'FechaEjecucion'\
                                  , 'Respuesta', 'prevPageToken', 'nextPageToken', 'ResultadosTotal', 'Resultadosxpagina']) # Dataframe de Informacion de Solicitudes

for consulta in consultas[ithConsulta:]: # For de consultas para la serie

  if (solicitud == solicitudmax+1):
    break

  for pais in paises[ithPais:]: # For del pais donde se busca

    if (solicitud == solicitudmax+1):
      break

    pageToken='' # Pagina especifica que se debe devolver => OJO DEBE DE IR EN ESTE LUGAR

    for pagina in range(1,paginasmax+1): # For de la paginacion

      solicitud = solicitud+1

      if (solicitud == solicitudmax+1):
        ptoPartida = [consulta,pais,pagina] # Punto de partida para volver a comenzar el motor de busqueda en produccion.
        break

      time.sleep(espera) # Tiempo de espera por cada solicitud
      r = requests.get(URIs+key+'&part='+part+'&fields='+fields+'&channelType='+channelType\
                       +'&maxResults='+maxResults+'&order='+order\
                       +'&pageToken='+pageToken+'&q='+consulta\
                       +'&regionCode='+pais+'&relevanceLanguage='\
                       +relevanceLanguage+'&safeSearch='+safeSearch) # Solicitud

      print('solicitud',solicitud,r.reason)

      if (r.reason == 'OK'): # Vemos si la solicitud fue exitosa
        d = dict(r.json())
        claves = list(d.keys())
        items = d.get('items')
        Data = pd.json_normalize(items).assign(query = consulta,regionCode = pais,solicitud = solicitud)
        Datac = pd.concat([Datac, Data], ignore_index = True)

        if ('nextPageToken' in claves): # Si hay siguiente pagina
          pageToken = d['nextPageToken'] # Definimos el proximo pagetoken
          if ('prevPageToken' in claves):
            prevPageToken = d['prevPageToken']
          else:
            prevPageToken = ''
          Metadatac = Metadatac.append({'Solicitud':solicitud,'maxResultados':maxResults,'Consulta':consulta\
                          ,'Pais':pais,'Lenguaje':relevanceLanguage,'FechaEjecucion':str(datetime.now())\
                          ,'Respuesta':r.reason,'nextPageToken':d['nextPageToken'],'prevPageToken':prevPageToken\
                          ,'ResultadosTotal':d['pageInfo']['totalResults'],'Resultadosxpagina':d['pageInfo']['resultsPerPage']}, ignore_index = True)
          print("Siguiente pagina")

        else: # Si no hay siguiente pagina
          nextPageToken ='' # Definimos el nextPageToken como vacio
          if ('prevPageToken' in claves):
            prevPageToken = d['prevPageToken']
          else:
            prevPageToken = ''
          Metadatac = Metadatac.append({'Solicitud':solicitud,'maxResultados':maxResults,'Consulta':consulta\
                          ,'Pais':pais,'Lenguaje':relevanceLanguage,'FechaEjecucion':str(datetime.now())\
                          ,'Respuesta':r.reason,'nextPageToken':nextPageToken,'prevPageToken':prevPageToken\
                          ,'ResultadosTotal':d['pageInfo']['totalResults'],'Resultadosxpagina':d['pageInfo']['resultsPerPage']}, ignore_index = True)
          print("No hay mas paginas")
          break
      else: # Vemos si la solicitud no fue exitosa
        print("No fue exitosa la solicitud", solicitud)
print("Busqueda Terminada")

In [ ]:
ptoPartida

In [ ]:
consultas.index(ptoPartida[0])

In [ ]:
Datac.info()

In [ ]:
Datac

In [ ]:
Metadatac.info()

In [ ]:
Metadatac

#### Estructurar y dar formato a Dataframe

In [ ]:
columnas = {'id.kind':'Recurso','id.playlistId':'idRecursoPL','snippet.publishedAt':'FechaCreacion','snippet.channelId':'idCanal'\
            ,'snippet.title':'Titulo','snippet.description':'Descripcion','snippet.channelTitle':'TituloCanal'\
            ,'snippet.liveBroadcastContent':'TransmisionVivo','id.videoId':'idRecursoVD','id.channelId':'idRecursoCH'\
            ,'query':'Consulta','regionCode':'Pais','solicitud':'Solicitud','snippet.publishTime':'TiempoPublicacion'}

columnasorden = ['Recurso', 'idRecursoVD', 'idRecursoPL', 'idRecursoCH', 'Titulo', 'Descripcion', 'TiempoPublicacion', 'FechaCreacion'\
                 , 'TransmisionVivo', 'idCanal', 'TituloCanal', 'Consulta', 'Pais', 'Solicitud']

Datac.rename(columns=columnas,inplace=True)
DATA=Datac.reindex(columns=columnasorden)
DATA

#### Extraccion de datos

Se generan 4 nuevas columnas (emoticones, hashtags, URLs y menciones) por cada columna de texto importante. (En este caso 8)

##### Extreaer emoticones

In [ ]:
def emojis(texto):
  if (len(texto) != 0):
    emoji_summary = adv.extract_emoji(texto)
    return emoji_summary['top_emoji']
  else:
    L=[]
    return L

In [ ]:
DATA['emojisTitulo']=DATA['Titulo'].apply(emojis)

In [ ]:
DATA['emojisTitulo']

In [ ]:
DATA['emojisDescripcion']=DATA['Descripcion'].apply(emojis)

In [ ]:
DATA['emojisDescripcion']

##### Extraer URLs

In [ ]:
extractor = URLExtract()
def URLs(texto):
  URL=extractor.find_urls(texto)
  return URL

In [ ]:
DATA['URLsDescripcion']=DATA['Descripcion'].apply(URLs)
DATA['URLsTitulo']=DATA['Titulo'].apply(URLs)

In [ ]:
DATA['URLsTitulo']

In [ ]:
DATA['URLsDescripcion']

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['Titulo'].apply(codigohex)
DATA['DescripcionModificado']=DATA['Descripcion'].apply(codigohex)

##### Extraer Hastags

In [ ]:
def hastags(texto):
  pattern='#\w+'
  hastag=regexp_tokenize(texto,pattern)
  return hastag

In [ ]:
DATA['hastagsDescripcion']=DATA['DescripcionModificado'].apply(hastags)
DATA['hastagsTitulo']=DATA['TituloModificado'].apply(hastags)

In [ ]:
DATA['hastagsTitulo']

In [ ]:
DATA['hastagsDescripcion']

##### Extraer menciones o etiquetas (puede ser necesario alguna modificacion para el caso de correos)

In [ ]:
def mencion(texto):
  pattern='@\w+'
  mencion=regexp_tokenize(texto,pattern)
  return mencion

In [ ]:
DATA['MencionesDescripcion']=DATA['Descripcion'].apply(mencion)
DATA['MencionesTitulo']=DATA['Titulo'].apply(mencion)

In [ ]:
DATA['MencionesTitulo']

In [ ]:
DATA['MencionesDescripcion']

In [ ]:
DATA.info()

#### Limpiar texto: (Generamos una copia llamada columname_modificado de una columna a partir del cual vamos a realizar la limpieza de datos)

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['Titulo'].apply(codigohex)
DATA['DescripcionModificado']=DATA['Descripcion'].apply(codigohex)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar Unicode (Es aca donde se eliminan las tildes-> Solucionado) NOTA: No se entiende muy bien que realiza este parte del codigo

In [ ]:
def dropunicode(texto):
  texto = unicodedata.normalize('NFD', texto)
  texto=texto.encode('ascii','ignore').decode('utf8')
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropunicode)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropunicode)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar secuencias de escape

In [ ]:
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\r', ' '))
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\n', ' '))
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\v', ' '))
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\t', ' '))
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\f', ' '))
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\b', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\r', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\n', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\v', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\t', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\f', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\b', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\r', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\n', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\v', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\t', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\f', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\b', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\r', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\n', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\v', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\t', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\f', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\b', ' '))

##### Eliminar URLs

In [ ]:
def dropURLs(texto):
  text = re.sub('https?://\S+|www\.\S+', ' ', texto)
  return text
def dropURL(texto):
  text = re.sub('http?://\S+|www\.\S+', ' ', texto)
  return text

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropURLs)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropURLs)
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropURL)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropURL)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar Emoticones

In [ ]:
from cleantext import clean

In [ ]:
from dataclasses import replace
def dropemoticons(texto):
  texto=clean(texto, no_emoji=True,lower=False)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropemoticons)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropemoticons)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar emails

In [ ]:
def dropemails(texto):
  texto=clean(texto, no_emails=True, lower=False)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropemails)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropemails)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar Menciones

In [ ]:
def dropmenciones(texto):
  texto = re.sub("@\S+", " ", texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropmenciones)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropmenciones)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Quitar caracteres repetidos


In [ ]:
def droppalabrasrepetidas(texto):
  texto = re.sub(r'(.)\1{3,}',r'\1', texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(droppalabrasrepetidas)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(droppalabrasrepetidas)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar Numeros

In [ ]:
def dropnumeros(texto):
  pattern = r'[0-9]'
  texto = re.sub(pattern, '', texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropnumeros)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropnumeros)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar \# de Hashtags

In [ ]:
def dropnumeral(texto):
  texto = re.sub(r'#', ' ', texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropnumeral)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropnumeral)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Dividir palabras adjuntas (CamelCase)

In [ ]:
def splitpalabras(texto):
  texto = " ".join([s for s in re.split("([A-Z][a-z]+[^A-Z]*)",texto) if s])
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(splitpalabras)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(splitpalabras)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar signos de puntuacion

In [ ]:
punct = set(string.punctuation)
punct

In [ ]:
def puntuacion(texto):
  sinpuntuacion="".join([ch for ch in texto if ch not in punct])
  return sinpuntuacion

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(puntuacion)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(puntuacion)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar espacios

In [ ]:
def dropespacios(texto):
  texto = re.sub('\s{2,}', " ", texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropespacios)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropespacios)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar tokens cortos

In [ ]:
def droppalabrascortas(texto):
  tokens = texto.split()
  clean_tokens = [t for t in tokens if len(t) > 1]
  clean_text = " ".join(clean_tokens)
  return clean_text

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(droppalabrascortas)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(droppalabrascortas)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Normalizar texto (Quitar mayusculas)

In [ ]:
def normatext(texto):
  textonormalizado=texto.lower()
  return textonormalizado

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(normatext)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(normatext)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

#### Estructurar y dar formato

In [ ]:
DATA.info()

In [ ]:
columnasorden = ['Recurso', 'idRecursoVD', 'idRecursoPL', 'idRecursoCH', 'Titulo', 'TituloModificado', 'emojisTitulo'\
                 , 'URLsTitulo', 'hastagsTitulo','MencionesTitulo', 'Descripcion', 'DescripcionModificado', 'emojisDescripcion', 'URLsDescripcion', 'hastagsDescripcion'\
                 , 'MencionesDescripcion','TiempoPublicacion', 'FechaCreacion', 'TransmisionVivo', 'idCanal', 'TituloCanal', 'Consulta', 'Pais', 'Solicitud']

DATA=DATA.reindex(columns=columnasorden)
DATA

#### Exportar Dataframe a CSV

In [ ]:
Metadatac.to_csv(PATH_TO_DIR_0+'/'+file_nameL+'.csv', index=False)
DATA.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)

## Concatenando bases de datos de busquedas relevantes y por numero de vistas

In [ ]:
# ================== OJO modificar para cada serie ==================

BusquedasSV = pd.read_csv ('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Search/\
Narcos/2022-09-12/RESULTS-SEARCH-2022-09-12 16:01:33.442696.csv')

BusquedasCR = pd.read_csv ('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Search/\
Narcos/2022-09-13SR1(2022-09-12)SR2(2022-09-13)/RESULTS-SEARCH-Relevance2022-09-13 13:33:05.564812.csv')

# ================== OJO modificar para cada serie ==================

In [ ]:
BusquedasSV

In [ ]:
BusquedasCR

In [ ]:
DATA=pd.concat([BusquedasSV,BusquedasCR])

In [ ]:
DATA

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Search/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today +'SV(2022-09-12)'+'CR(2022-09-13)'
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'SEARCH-'+str(datetime.now()) # +'Relevance' ========> OJO modificar si es por relevancia o por numero de vistas

In [ ]:
DATA.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)